# 02 — Classical Markov-Switching Baseline

Fits the Hamilton (1989) Markov-switching model via EM algorithm. This is our interpretable baseline before introducing Bayesian estimation.

**Prerequisites:** Run `01_eda.ipynb` first.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.data.loader import get_price_data, load_config
from src.models.markov_switching import HamiltonMarkovSwitching, plot_regime_probabilities, plot_regime_statistics
from src.utils.metrics import regime_performance_breakdown

%matplotlib inline
config = load_config('../config/settings.yaml')
print("Ready")

In [ ]:
prices = get_price_data(config)
returns = np.log(prices['index'] / prices['index'].shift(1)).dropna()
print(f"Returns: {len(returns)} observations | {returns.index[0].date()} → {returns.index[-1].date()}")
print(f"Annualised return: {returns.mean() * 252:.2%}")
print(f"Annualised vol:    {returns.std() * np.sqrt(252):.2%}")

## Fit 2-State Model (Bull / Bear)

In [ ]:
model_2 = HamiltonMarkovSwitching(n_regimes=2, config=config)
model_2.cfg['regimes']['labels'] = {0: 'Bull', 1: 'Bear'}
model_2.fit(returns)

print("\nExpected durations:")
print(model_2.expected_durations())

## Fit 3-State Model (Full Framework)

In [ ]:
model_3 = HamiltonMarkovSwitching(n_regimes=3, config=config)
model_3.fit(returns)

print("\nExpected durations:")
print(model_3.expected_durations())

print("\nRegime classification counts:")
print(model_3.regime_classification.value_counts())

In [ ]:
# Override config paths for notebook context
config['paths']['figures'] = '../results/figures/'
config['paths']['processed_data'] = '../data/processed/'

plot_regime_probabilities(model_3, prices['index'], config, save=True)

In [ ]:
plot_regime_statistics(model_3, returns, config)

## Performance by Regime

In [ ]:
breakdown = regime_performance_breakdown(returns, model_3.regime_classification)
print("\nPerformance breakdown by regime:")
print(breakdown.to_string())

In [ ]:
# Save for use in backtest notebook
model_3.save_results(config)
print("Classical model results saved.")

## Model Comparison: AIC / BIC

In [ ]:
print("Model comparison:")
print(f"  2-state: AIC={model_2.result.aic:.1f} | BIC={model_2.result.bic:.1f}")
print(f"  3-state: AIC={model_3.result.aic:.1f} | BIC={model_3.result.bic:.1f}")
print()
print("Lower AIC/BIC = better fit. But consider interpretability too.")